# Fraud / Anomaly Detection
## Notebook 07 — Isolation Forest

### Objective
Train the primary tree-based anomaly detector and save its scores and model.

## Imports and sample

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.fraud_detection import load_features_labels, evaluate_scores, save_results
X, y = load_features_labels(ROOT)
RANDOM_STATE = 42
print('Features:', X.shape, '| fraud:', int(y.sum()))
from src.fraud_detection import stratified_working_sample
from sklearn.ensemble import IsolationForest
import joblib
Xw, yw = stratified_working_sample(X, y, max_rows=60000, random_state=RANDOM_STATE)

**Interpretation.** A bounded stratified working set makes experiments fast while retaining every rare fraud example.

**ML decision.** Isolation Forest is the primary scalable candidate.

## Train, score, and save

In [ ]:
model = IsolationForest(n_estimators=200, contamination='auto', random_state=RANDOM_STATE, n_jobs=-1)
model.fit(Xw)
scores = -model.score_samples(Xw)
result = evaluate_scores(yw, scores, 'Isolation Forest')
joblib.dump(model, ROOT/'models/isolation_forest.joblib')
pd.DataFrame({'row_index': Xw.index, 'label': yw, 'score': scores}).to_csv(ROOT/'reports/isolation_forest_scores.csv', index=False)
result

**Interpretation.** Higher negative score-sample values indicate observations isolated in fewer splits.

**ML decision.** Use PR-AUC and recall to decide whether this remains the primary model.